# Colab 2 · Orchestration Patterns in Depth
### Day 19 — Agent Orchestration with AutoGen Studio & Semantic Kernel

Colab 1 used the simplest pattern (RoundRobin). Now you'll wire the **same research team four different ways** and watch how the *control flow* changes — then peek at the **same idea in Semantic Kernel**.

**You will build:**
1. **SelectorGroupChat** — an LLM decides who speaks next.
2. **Swarm** — agents hand off to each other directly.
3. **GraphFlow** — a deterministic researcher → writer → reviewer graph.
4. A **function tool** the researcher calls to delegate real work.
5. A **Semantic Kernel** sequential-orchestration mini-example.

⏱️ ~60 min including the extension tasks at the end.

> The patterns are the lesson. AutoGen, Semantic Kernel and the Microsoft Agent Framework all expose this same family — RoundRobin/Sequential, Selector/GroupChat, Swarm/Handoff, Graph, Magentic.

## 0 · Setup

In [ ]:
%pip install -q -U "autogen-agentchat" "autogen-ext[openai]"
print("AutoGen installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 16.4 MB/s eta 0:00:00
AutoGen installed.


In [3]:
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
print("Ready.")

Ready.


### Three specialists we'll reuse

Notice the **descriptions** — Selector and Swarm route work based on them, so they have to be specific and non-overlapping.

In [4]:
def make_specialists():
    planner = AssistantAgent(
        name="planner",
        model_client=model_client,
        description="Breaks a topic into 2-3 concrete sub-questions to research.",
        system_message="You plan research. Given a topic, list 2-3 specific sub-questions. Keep it short.",
    )
    researcher = AssistantAgent(
        name="researcher",
        model_client=model_client,
        description="Answers factual sub-questions with concise bullet points.",
        system_message="You answer the planner's sub-questions with short factual bullets.",
    )
    writer = AssistantAgent(
        name="writer",
        model_client=model_client,
        description="Turns research bullets into a tight 4-sentence summary, ending with APPROVE.",
        system_message="Write a tight 4-sentence summary from the research. End your message with APPROVE.",
    )
    return planner, researcher, writer

print("Specialist factory ready.")

Specialist factory ready.


## 1 · SelectorGroupChat — let an LLM route

Instead of a fixed order, a **SelectorGroupChat** uses a model to pick *who should act next* based on the conversation and each agent's `description`. Good when the next best speaker depends on what just happened.

Key knobs: it needs its own `model_client` to do the routing, and `allow_repeated_speaker=False` stops one agent from monopolising the floor.

In [5]:
from autogen_agentchat.teams import SelectorGroupChat

planner, researcher, writer = make_specialists()
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

selector_team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,        # the "router" brain
    termination_condition=termination,
    allow_repeated_speaker=False,
)

await Console(selector_team.run_stream(
    task="Topic: why are reusable cups better than disposable ones?"
))

---------- TextMessage (user) ----------
Topic: why are reusable cups better than disposable ones?
---------- TextMessage (planner) ----------
1. What are the environmental impacts of producing and disposing of reusable cups compared to disposable ones?  
2. How do the long-term costs of using reusable cups compare to those of disposable cups for consumers?  
3. What influence do reusable cups have on consumer behavior regarding sustainability and waste reduction?
---------- TextMessage (researcher) ----------
1. Environmental impacts:
   - Reusable cups reduce waste by minimizing the number of single-use cups sent to landfills.
   - Disposable cup production often involves higher resource use (e.g., trees, plastic, water) and generates more greenhouse gas emissions.
   - Reusables can be made from sustainable materials (like stainless steel or glass) and are often produced with eco-friendly practices.
   - Disposal of disposable cups can lead to pollution, especially if they are not c

TaskResult(messages=[TextMessage(id='ac6a2a15-df7d-4525-93d2-2bea2a05e275', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 47, 50, 443350, tzinfo=datetime.timezone.utc), content='Topic: why are reusable cups better than disposable ones?', type='TextMessage'), TextMessage(id='117b447c-a02f-4edd-a45d-925493e30b42', source='planner', models_usage=RequestUsage(prompt_tokens=45, completion_tokens=59), metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 47, 58, 194423, tzinfo=datetime.timezone.utc), content='1. What are the environmental impacts of producing and disposing of reusable cups compared to disposable ones?  \n2. How do the long-term costs of using reusable cups compare to those of disposable cups for consumers?  \n3. What influence do reusable cups have on consumer behavior regarding sustainability and waste reduction?', type='TextMessage'), TextMessage(id='f28dcfef-5202-424a-b469-02dd5a9d87d6', source='researcher', models_usage

Look at the speaker order in the transcript — it was **chosen at runtime**, not fixed. That's the difference from RoundRobin.

## 2 · Swarm — agents hand off to each other

In a **Swarm**, control is decentralised: each agent declares who it can **hand off** to via `handoffs=[...]`, and passes control with a handoff message. There's no central router — the agents themselves decide.

We'll build a tiny triage flow: a `triage` agent routes to either `billing` or `tech`, and those can hand back to the user when done.

In [6]:
from autogen_agentchat.teams import Swarm
from autogen_agentchat.conditions import HandoffTermination

triage = AssistantAgent(
    name="triage",
    model_client=model_client,
    handoffs=["billing", "tech"],
    description="Front desk: routes the user to the right specialist.",
    system_message="Decide if the request is about billing or tech, then hand off to that agent.",
)
billing = AssistantAgent(
    name="billing",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles billing and refund questions.",
    system_message="Answer the billing question. If it's not billing, hand back to triage.",
)
tech = AssistantAgent(
    name="tech",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles technical troubleshooting.",
    system_message="Answer the tech question concisely, then say DONE.",
)

swarm = Swarm(
    participants=[triage, billing, tech],          # Swarm starts with the first agent
    termination_condition=TextMentionTermination("DONE") | MaxMessageTermination(8),
)

await Console(swarm.run_stream(task="My app keeps crashing when I open the camera."))

---------- TextMessage (user) ----------
My app keeps crashing when I open the camera.
---------- ToolCallRequestEvent (triage) ----------
[FunctionCall(id='call_sl6MUWUAoyDspF1b17kL6jk8', arguments='{}', name='transfer_to_tech')]
---------- ToolCallExecutionEvent (triage) ----------
[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='transfer_to_tech', call_id='call_sl6MUWUAoyDspF1b17kL6jk8', is_error=False)]
---------- HandoffMessage (triage) ----------
Transferred to tech, adopting the role of tech immediately.
---------- ToolCallRequestEvent (tech) ----------
[FunctionCall(id='call_OqfNrjvJCf44hv3IQNwOY9VN', arguments='{}', name='transfer_to_triage')]
---------- ToolCallExecutionEvent (tech) ----------
[FunctionExecutionResult(content='Transferred to triage, adopting the role of triage immediately.', name='transfer_to_triage', call_id='call_OqfNrjvJCf44hv3IQNwOY9VN', is_error=False)]
---------- HandoffMessage (tech) ----------
Trans

TaskResult(messages=[TextMessage(id='22f6a42d-3df1-4c76-bdb6-867674f95b1f', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 48, 18, 905198, tzinfo=datetime.timezone.utc), content='My app keeps crashing when I open the camera.', type='TextMessage'), ToolCallRequestEvent(id='4f04f85c-c102-420e-92c1-93e4d112f5bb', source='triage', models_usage=RequestUsage(prompt_tokens=82, completion_tokens=12), metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 48, 19, 867541, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_sl6MUWUAoyDspF1b17kL6jk8', arguments='{}', name='transfer_to_tech')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='e98bc227-970d-4ab7-a589-99f60027808b', source='triage', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 48, 19, 871953, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='tra

Watch for the **HandoffMessage** in the transcript — that's one agent explicitly delegating to another. `HandoffTermination(target="user")` is another common stop condition when an agent hands control back to a human.

## 3 · GraphFlow — a deterministic workflow

When you need the **same path every time** (auditable, reproducible), use **GraphFlow**. You declare nodes and directed edges with `DiGraphBuilder`; execution follows the graph exactly.

Here: `planner → researcher → writer`, a fixed pipeline with no LLM routing brain.

In [7]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

planner, researcher, writer = make_specialists()

builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer)
builder.add_edge(planner, researcher).add_edge(researcher, writer)
graph = builder.build()

flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
)

await Console(flow.run_stream(task="Topic: the benefits of cycling to work."))

---------- TextMessage (user) ----------
Topic: the benefits of cycling to work.
---------- TextMessage (planner) ----------
1. How does cycling to work impact physical health compared to other commuting methods?  
2. What are the environmental benefits of increased cycling among commuters?  
3. How does cycling to work affect mental well-being and job satisfaction?
---------- TextMessage (researcher) ----------
1. **Impact on Physical Health:**
   - Improves cardiovascular fitness and overall endurance.
   - Aids in weight management and reduces obesity risk.
   - Strengthens muscles and improves joint flexibility.
   - Can lower the risk of chronic illnesses (e.g., diabetes, heart disease).
   - Provides a consistent physical activity routine.

2. **Environmental Benefits:**
   - Reduces greenhouse gas emissions and improves air quality.
   - Lowers traffic congestion and noise pollution.
   - Decreases reliance on fossil fuels and non-renewable energy sources.
   - Contributes to le

TaskResult(messages=[TextMessage(id='7304ce44-3436-49d3-94e7-da6a709ef374', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 48, 30, 863438, tzinfo=datetime.timezone.utc), content='Topic: the benefits of cycling to work.', type='TextMessage'), TextMessage(id='92a623b0-21dd-46cd-baf2-13537a66988e', source='planner', models_usage=RequestUsage(prompt_tokens=43, completion_tokens=46), metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 48, 32, 994148, tzinfo=datetime.timezone.utc), content='1. How does cycling to work impact physical health compared to other commuting methods?  \n2. What are the environmental benefits of increased cycling among commuters?  \n3. How does cycling to work affect mental well-being and job satisfaction?', type='TextMessage'), TextMessage(id='8ef9e3d6-0171-48b6-acbc-1d9c5cd63a92', source='researcher', models_usage=RequestUsage(prompt_tokens=86, completion_tokens=211), metadata={}, created_at=datetime.datetime(20

GraphFlow gives you **determinism**: the order is guaranteed by the graph, not decided by a model. That's exactly what you want for a compliance-sensitive or repeatable pipeline.

## 4 · A function tool the researcher can call

Delegation isn't only agent-to-agent — an agent can delegate to **code** via a tool. Define a plain Python function, pass it in `tools=[...]`, and the agent will call it when useful. (Here it's a stub; swap in a real search API at home.)

In [8]:
def web_search(query: str) -> str:
    """Look up a query and return a short text snippet. (Stub for the workshop.)"""
    canned = {
        "reusable cup co2": "A reusable cup typically breaks even vs. disposables after ~20-100 uses.",
        "default": "No exact match; returning a generic note that reusable goods amortise their footprint with use.",
    }
    return canned.get(query.lower().strip(), canned["default"])

researcher_with_tool = AssistantAgent(
    name="researcher",
    model_client=model_client,
    tools=[web_search],
    description="Researches facts, calling web_search when it needs evidence.",
    system_message="Use the web_search tool to find a figure, then report it in one bullet. End with APPROVE.",
)

from autogen_agentchat.teams import RoundRobinGroupChat
tool_team = RoundRobinGroupChat(
    [researcher_with_tool],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(4),
)
await Console(tool_team.run_stream(task="Find a figure on reusable cup CO2 break-even and report it."))

---------- TextMessage (user) ----------
Find a figure on reusable cup CO2 break-even and report it.
---------- ToolCallRequestEvent (researcher) ----------
[FunctionCall(id='call_xlbTXd0Q0nucAIVhI5TxO9WN', arguments='{"query":"reusable cup CO2 break-even figure"}', name='web_search')]
---------- ToolCallExecutionEvent (researcher) ----------
[FunctionExecutionResult(content='No exact match; returning a generic note that reusable goods amortise their footprint with use.', name='web_search', call_id='call_xlbTXd0Q0nucAIVhI5TxO9WN', is_error=False)]
---------- ToolCallSummaryMessage (researcher) ----------
No exact match; returning a generic note that reusable goods amortise their footprint with use.
---------- TextMessage (researcher) ----------
Reusable cups generally amortize their carbon footprint with multiple uses, and while specific figures may vary, it's often cited that a reusable cup can offset the CO2 emissions of around 100-200 single-use cups over its lifetime. APPROVE


TaskResult(messages=[TextMessage(id='7f92b469-d65b-4663-b168-c772aa4f787c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 48, 42, 987914, tzinfo=datetime.timezone.utc), content='Find a figure on reusable cup CO2 break-even and report it.', type='TextMessage'), ToolCallRequestEvent(id='bbcc2181-a208-4098-8cb0-bd58c1163c86', source='researcher', models_usage=RequestUsage(prompt_tokens=97, completion_tokens=21), metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 48, 43, 875896, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_xlbTXd0Q0nucAIVhI5TxO9WN', arguments='{"query":"reusable cup CO2 break-even figure"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='151cef69-06f5-4de6-b427-630e357057b7', source='researcher', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 48, 43, 878550, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='No exact ma

The transcript shows a **ToolCall** and its result — the agent delegated part of its job to your function. In production you'd scope tools to least privilege and validate their arguments.

## 5 · The same idea in Semantic Kernel

Semantic Kernel expresses these patterns too — its **Sequential** orchestration is the SK analogue of RoundRobin/GraphFlow-in-a-line. The code below shows the *shape* of SK agent orchestration.

> ⚠️ SK's agent-orchestration API is newer and evolving (and SK is in maintenance mode heading into the Microsoft Agent Framework). If an import path has moved, check the official Semantic Kernel docs — the **concept** is what transfers, not the exact symbol names.

In [9]:
%pip install -q -U semantic-kernel
print("Semantic Kernel installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.3/929.3 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.9/217.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [10]:
# The shape of an SK sequential orchestration: two agents, output of one feeds the next.
# Wrapped in try/except because SK's orchestration symbols move between versions.
import asyncio

try:
    from semantic_kernel.agents import ChatCompletionAgent
    from semantic_kernel.agents.orchestration.sequential import SequentialOrchestration
    from semantic_kernel.agents.runtime import InProcessRuntime
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

    service = OpenAIChatCompletion(ai_model_id="gpt-4o-mini")

    sk_writer = ChatCompletionAgent(
        name="writer", service=service,
        instructions="Write one short paragraph on the given topic.",
    )
    sk_editor = ChatCompletionAgent(
        name="editor", service=service,
        instructions="Tighten the paragraph you receive into two crisp sentences.",
    )

    orchestration = SequentialOrchestration(members=[sk_writer, sk_editor])
    runtime = InProcessRuntime()
    runtime.start()

    result = await orchestration.invoke(
        task="The benefits of walking meetings.", runtime=runtime
    )
    print(await result.get())
    await runtime.stop_when_idle()

except Exception as e:
    print("SK orchestration symbols may have moved in your installed version.")
    print("Concept: members=[writer, editor] run in sequence, output -> input.")
    print("Check https://learn.microsoft.com/semantic-kernel for the current API.")
    print("Error was:", type(e).__name__, e)

Walking meetings provide a refreshing alternative to traditional discussions by promoting physical activity and mental clarity, which enhances creativity and problem-solving. This dynamic setting fosters open communication and collaboration while offering health benefits like improved cardiovascular health and reduced stress.


Notice the **identical mental model**: a list of agents, run in order, each one's output feeding the next. RoundRobin (AutoGen) ≈ Sequential (SK) ≈ a linear GraphFlow. Learn it once.

---
## 🚀 Extension tasks

### Extension 1 — Write a custom selector function
`SelectorGroupChat` accepts a `selector_func` that overrides the LLM router with your own logic. Write a function that **forces** `planner` to go first, then lets the model choose. (Signature: it receives the message history and returns the next speaker's name, or `None` to defer to the model.)

### Extension 2 — Add a conditional GraphFlow edge
Extend the graph from §3 with a **reviewer** node and a **conditional edge**: if the writer's output contains the word `REVISE`, loop back to the writer; otherwise finish. Use `DiGraphBuilder`'s conditional-edge support and a stop condition so it can't loop forever.

### Extension 3 — Nest a team inside a graph node
A node in a GraphFlow can itself be a **team**. Replace the single `researcher` node with a 2-agent `RoundRobinGroupChat` (researcher + fact-checker) and wire that team in as one node. This is *composition*: patterns nest inside patterns.

Scaffolds below.

In [11]:
# === Extension 1 scaffold: custom selector_func ===
def force_planner_first(messages):
    # Return an agent name (str) to force a speaker, or None to let the model decide.
    if len(messages) <= 1:
        return "planner"
    return None

planner, researcher, writer = make_specialists()
team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,
    selector_func=force_planner_first,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8),
)
await Console(team.run_stream(task="Topic: benefits of a standing desk."))

---------- TextMessage (user) ----------
Topic: benefits of a standing desk.
---------- TextMessage (planner) ----------
1. How does using a standing desk affect productivity and focus in the workplace?
2. What are the physical health benefits associated with transitioning to a standing desk?
3. How does standing desk usage impact employee morale and job satisfaction?
---------- TextMessage (researcher) ----------
1. **Productivity and Focus:**
   - Standing desks can enhance alertness and energy levels.
   - Some studies suggest improved concentration and reduced fatigue.
   - Users may experience less distraction and improved cognitive functioning.

2. **Physical Health Benefits:**
   - Reduced risk of weight gain and obesity due to increased energy expenditure.
   - Decreased risk of chronic diseases, such as heart disease and diabetes.
   - Alleviation of back and neck pain associated with prolonged sitting.
   - Improved posture and reduced incidence of musculoskeletal disorders.


TaskResult(messages=[TextMessage(id='e82ca3dc-99b0-4256-9d03-4a11a87ed24f', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 49, 40, 611704, tzinfo=datetime.timezone.utc), content='Topic: benefits of a standing desk.', type='TextMessage'), TextMessage(id='ce00f3ed-b75d-4634-85af-cb6a46e058c2', source='planner', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=46), metadata={}, created_at=datetime.datetime(2026, 6, 22, 15, 49, 41, 642237, tzinfo=datetime.timezone.utc), content='1. How does using a standing desk affect productivity and focus in the workplace?\n2. What are the physical health benefits associated with transitioning to a standing desk?\n3. How does standing desk usage impact employee morale and job satisfaction?', type='TextMessage'), TextMessage(id='2d6fdbaa-494a-4d47-8949-a31519341d0d', source='researcher', models_usage=RequestUsage(prompt_tokens=85, completion_tokens=174), metadata={}, created_at=datetime.datetime(

In [18]:
# Define the reviewer specialist
reviewer = AssistantAgent(
    name="reviewer",
    model_client=model_client,
    description="Reviews summaries for accuracy.",
    system_message="Check if the summary is accurate. If it needs work, say REVISE. If it is good, say APPROVE.",
)

# Terminal node to satisfy graph validation
final_report = AssistantAgent(
    name="final_report",
    model_client=model_client,
    system_message="The process is complete. Just repeat the final summary."
)

# === Extension 2: conditional edge with termination ===
builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer).add_node(reviewer).add_node(final_report)

builder.add_edge(planner, researcher)
builder.add_edge(researcher, writer)
builder.add_edge(writer, reviewer)

# Conditional loop and exit
builder.add_edge(reviewer, writer, condition=lambda msg: "REVISE" in msg.content.upper())
builder.add_edge(reviewer, final_report, condition=lambda msg: "APPROVE" in msg.content.upper())

graph = builder.build()

# CRITICAL: Added termination_condition for the cyclic graph
flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(10)
)

# Run the flow
await Console(flow.run_stream(task="Topic: history of the espresso machine."))

---------- TextMessage (user) ----------
Topic: history of the espresso machine.
---------- TextMessage (planner) ----------
1. What were the key inventions that led to the development of the first espresso machine?
2. How did espresso machines evolve throughout the 20th century?
3. What role did the espresso machine play in shaping coffee culture in different regions?
---------- TextMessage (researcher) ----------
1. **Key Inventions Leading to Espresso Machine Development:**
   - **The Steam Engine (18th Century):** Provided the concept of using steam pressure to brew coffee.
   - **Pavoni's Espresso Machine (1905):** Introduced the first commercial espresso machine using steam pressure for extraction.
   - **A. N. R. Bezzera's Machine (1901):** Developed a machine allowing for quicker coffee brewing under pressure.

2. **Evolution of Espresso Machines Throughout the 20th Century:**
   - **1910s:** Introduction of commercial machines capable of making multiple shots.
   - **1930s:** 

TaskResult(messages=[TextMessage(id='85fc8dad-fdc3-44fc-84de-e954adf3d0c1', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 16, 33, 27, 941338, tzinfo=datetime.timezone.utc), content='Topic: history of the espresso machine.', type='TextMessage'), TextMessage(id='01df8b18-8731-4303-8b0b-c5085f139188', source='planner', models_usage=RequestUsage(prompt_tokens=105, completion_tokens=49), metadata={}, created_at=datetime.datetime(2026, 6, 22, 16, 33, 30, 162109, tzinfo=datetime.timezone.utc), content='1. What were the key inventions that led to the development of the first espresso machine?\n2. How did espresso machines evolve throughout the 20th century?\n3. What role did the espresso machine play in shaping coffee culture in different regions?', type='TextMessage'), TextMessage(id='fe965d2f-1858-417a-b132-3173186e10c9', source='researcher', models_usage=RequestUsage(prompt_tokens=331, completion_tokens=297), metadata={}, created_at=datetime.dateti

In [36]:
import autogen_agentchat
from autogen_agentchat.base import ChatAgent, Response, TaskResult
from autogen_agentchat.messages import TextMessage
from typing import List, AsyncGenerator, Sequence, Type, Union, Any
from autogen_agentchat.teams import RoundRobinGroupChat, DiGraphBuilder, GraphFlow
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

# Fallback wrapper implementing required properties and methods
class CustomTeamAgent(ChatAgent):
    def __init__(self, name: str, team):
        self._name = name
        self._description = f"A team called {name}"
        self._team = team
        super().__init__(self._name, self._description)

    @property
    def name(self) -> str:
        return self._name

    @property
    def description(self) -> str:
        return self._description

    @property
    def produced_message_types(self) -> List[Type[Any]]:
        return [TextMessage]

    async def on_messages(self, messages: Sequence[Any], cancellation_token: Any) -> Response:
        result = await self._team.run(task=messages[-1].content)
        return Response(chat_message=result.messages[-1])

    async def on_messages_stream(self, messages: Sequence[Any], cancellation_token: Any) -> AsyncGenerator[Union[Any, Response], None]:
        last_message = None
        # Stream from the inner team
        async for chunk in self._team.run_stream(task=messages[-1].content):
             if isinstance(chunk, TaskResult):
                 # Store the final message from the result to yield as a Response later
                 if chunk.messages:
                     last_message = chunk.messages[-1]
             else:
                 # Yield intermediate messages (like researcher's findings)
                 yield chunk

        # CRITICAL: An agent stream MUST yield a Response object at the end to avoid RuntimeError
        if last_message:
            yield Response(chat_message=last_message)

    async def on_reset(self, cancellation_token: Any) -> None:
        await self._team.reset()

    async def close(self) -> None: pass
    async def load_state(self, state: dict) -> None: pass
    async def save_state(self) -> dict: return {}
    async def on_pause(self, cancellation_token: Any) -> None: pass
    async def on_resume(self, cancellation_token: Any) -> None: pass

# Ensure base specialists exist
planner, researcher, writer = make_specialists()

# Define the fact_checker specialist
fact_checker = AssistantAgent(
    name="fact_checker",
    model_client=model_client,
    description="Verifies facts against a high standard.",
    system_message="Look at the research provided. If any facts look suspicious, point them out. Otherwise, say VERIFIED.",
)

# 1. Create the sub-team
inner_team = RoundRobinGroupChat(
    [researcher, fact_checker],
    termination_condition=TextMentionTermination("VERIFIED") | MaxMessageTermination(4),
)

# 2. Wrap the team in our CustomTeamAgent
research_team_node = CustomTeamAgent(
    name="research_team",
    team=inner_team
)

# Create the graph
try:
    builder = DiGraphBuilder()
    builder.add_node(planner).add_node(research_team_node).add_node(writer)
    builder.add_edge(planner, research_team_node).add_edge(research_team_node, writer)

    graph = builder.build()
    flow = GraphFlow(participants=builder.get_participants(), graph=graph)

    print("Nested flow configured successfully. Running...")
    # Run the nested flow
    await Console(flow.run_stream(task="Topic: how solar panels work."))
except Exception as e:
    import traceback
    print(f"Configuration error: {e}")
    traceback.print_exc()

Nested flow configured successfully. Running...
---------- TextMessage (user) ----------
Topic: how solar panels work.
---------- TextMessage (planner) ----------
1. What are the key components of a solar panel and their functions?
2. How do photovoltaic cells convert sunlight into electricity?
3. What factors affect the efficiency of solar panels in energy generation?
---------- TextMessage (user) ----------
1. What are the key components of a solar panel and their functions?
2. How do photovoltaic cells convert sunlight into electricity?
3. What factors affect the efficiency of solar panels in energy generation?
---------- TextMessage (researcher) ----------
1. **Key Components of a Solar Panel**:
   - **Photovoltaic Cells**: Convert sunlight into electricity.
   - **Glass Layer**: Protects the cells and allows sunlight to penetrate.
   - **Anti-Reflective Coating**: Reduces light reflection to enhance absorption.
   - **Backsheet**: Provides insulation and protects the internal comp

## Recap

You orchestrated one research team **five ways** and saw exactly how control flow differs:

* **Selector** — LLM picks the next speaker (dynamic).
* **Swarm** — agents hand off to each other (decentralised).
* **GraphFlow** — a fixed, auditable graph (deterministic).
* **Tools** — an agent delegates work to a function.
* **Semantic Kernel** — the same Sequential idea in the enterprise SDK.

Pair this with the decision matrix from the slides, then tackle the **capstone**: build your own delegating team in AutoGen Studio.

In [37]:
await model_client.close()
print("Client closed. On to the capstone!")

Client closed. On to the capstone!
